# Online Shoppers Purchasing Intention — Model Training

End-to-end training and evaluation of five classification models on the UCI **Online Shoppers Purchasing Intention** dataset.

This notebook reproduces the training done by the scripts in `model/`. Run it top-to-bottom on the BITS Virtual Lab and capture a screenshot of the output for the submission.

**Models:** Logistic Regression · Decision Tree · kNN · Naive Bayes · Random Forest  
**Metrics:** Accuracy · AUC · Precision · Recall · F1 · MCC

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report,
)

RANDOM_STATE = 42

## 2. Load the dataset

In [ ]:
df = pd.read_csv('../data/online_shoppers_intention.csv')
print('Shape:', df.shape)
print('Positive rate (Revenue=True): %.3f' % df['Revenue'].mean())
df.head()

In [ ]:
df['Revenue'].value_counts()

## 3. Features and preprocessing

Numeric features are standardised (important for kNN and Logistic Regression); categorical features are one-hot encoded. Both steps live inside each model pipeline.

In [ ]:
TARGET = 'Revenue'

NUMERIC_FEATURES = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
]
CATEGORICAL_FEATURES = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend',
]

def build_preprocessor():
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ])

## 4. Train/test split (stratified 80/20)

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print('Train:', X_train.shape[0], '| Test:', X_test.shape[0])

## 5. Define the five models

`class_weight='balanced'` is used where supported because the target is imbalanced (~15% positive). kNN uses k ≈ √N (forced odd).

In [ ]:
k = int(np.sqrt(X_train.shape[0]))
if k % 2 == 0:
    k += 1
print('kNN k =', k)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, class_weight='balanced', random_state=RANDOM_STATE),
    'kNN': KNeighborsClassifier(n_neighbors=k),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
}

models = {name: Pipeline([('preprocess', build_preprocessor()), ('classifier', clf)])
          for name, clf in classifiers.items()}

## 6. Train and evaluate all models

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_proba),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred),
    }

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    results[name] = evaluate(pipeline, X_test, y_test)
    print(f'Trained: {name}')

results_df = pd.DataFrame(results).transpose().round(4)
results_df

## 7. Comparison of all models

In [ ]:
ax = results_df.plot(kind='bar', figsize=(11, 5))
ax.set_title('Model comparison on the held-out test set')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.legend(loc='lower right', ncol=3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

winner = results_df['MCC'].idxmax()
print('Best model by MCC:', winner, '(MCC = %.4f)' % results_df.loc[winner, 'MCC'])

## 8. Confusion matrix for the best model

In [ ]:
best_model = models[winner]
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No purchase', 'Purchase'],
            yticklabels=['No purchase', 'Purchase'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix — {winner}')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['No purchase', 'Purchase']))